# Comprehensive Interactive EDA: Orders & Relationships
**Datathon 2026 - Round 1**

This notebook explores the `orders.csv` dataset and its relationships with customers, geography, and revenue. We follow the 4-level analysis framework: **Descriptive**, **Diagnostic**, **Predictive**, and **Prescriptive**.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import time
import os

# Force plotly to use a template
import plotly.io as pio
pio.templates.default = "plotly_white"

# Project configuration
DATA_DIR = Path("../data/raw")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 1. Data Loading & Merging
We join `orders` with `customers`, `geography`, and `order_items` to enable relationship analysis. 
**Speed Optimization**: Using `engine='pyarrow'` and vectorized calculations.

In [2]:
start_time = time.time()
print("Loading datasets with PyArrow engine...")

orders = pd.read_csv(DATA_DIR / "orders.csv", engine='pyarrow')
customers = pd.read_csv(DATA_DIR / "customers.csv", engine='pyarrow')
geography = pd.read_csv(DATA_DIR / "geography.csv", engine='pyarrow')
order_items = pd.read_csv(DATA_DIR / "order_items.csv", engine='pyarrow', dtype={'promo_id': str, 'promo_id_2': str})

orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['year_month'] = orders['order_date'].dt.to_period('M').astype(str)
order_items['revenue'] = order_items['unit_price'] * order_items['quantity']

print("Merging datasets...")
df = orders.merge(customers[['customer_id', 'age_group', 'gender', 'city', 'acquisition_channel']], on='customer_id', how='left')
df = df.merge(geography[['zip', 'region', 'district']], on='zip', how='left')

order_summary = order_items.groupby('order_id').agg(
    total_revenue=('revenue', 'sum'),
    total_items=('quantity', 'sum')
).reset_index()

df = df.merge(order_summary, on='order_id', how='left')
end_time = time.time()

print(f"Initial data merged. Total records: {len(df)}")
print(f"Total Loading & Processing Time: {end_time - start_time:.2f} seconds")

Loading datasets with PyArrow engine...
Merging datasets...
Initial data merged. Total records: 646945
Total Loading & Processing Time: 0.51 seconds


## 2. Temporal Analysis
### Monthly Order Volume Trend

In [3]:
monthly_growth = df.groupby('year_month').agg(order_count=('order_id', 'count')).reset_index()
fig = px.line(monthly_growth, x='year_month', y='order_count', 
              title='Monthly Order Volume Trend (2012 - 2022)')
fig.write_image(PLOT_DIR / "01_monthly_orders.png")
fig.show()

## 3. Order Status Funnel

In [4]:
status_dist = df['order_status'].value_counts().reset_index()
status_dist.columns = ['status', 'count']
fig = px.funnel(status_dist, x='count', y='status', title='Order Status Flow Analysis')
fig.write_image(PLOT_DIR / "02_status_funnel.png")
fig.show()

## 4. Demographic Sunburst

In [5]:
sun_df = df.dropna(subset=['age_group', 'gender', 'device_type'])
fig = px.sunburst(sun_df, path=['age_group', 'gender', 'device_type'], 
                  values='total_revenue', title='Revenue Distribution')
fig.write_image(PLOT_DIR / "03_demographic_sunburst.png")
fig.show()

## 5. Channel Efficiency

In [6]:
channel_stats = df.groupby('acquisition_channel').agg(avg_order_value=('total_revenue', 'mean')).reset_index()
fig = px.bar(channel_stats, x='acquisition_channel', y='avg_order_value', title='AOV by Channel')
fig.write_image(PLOT_DIR / "04_channel_aov.png")
fig.show()